LING 430 Fall 2026 

lecture 07a

in-class activity

#### setup

In [ ]:
import re
import random
import nltk
from urllib.request import urlopen

#### a model from a text

In [ ]:
def tokenize(text):
    text = text.replace("’", "'")
    return re.findall(r"[\w@#']+|[.,!?;:]", text)

def gutenberg(number):
    url = f"https://www.gutenberg.org/files/{number}/{number}-0.txt"
    raw = urlopen(url).read().decode("utf-8-sig")
    start = raw.find("\n", raw.find("*** START")) + 1
    return raw[start:raw.find("*** END")].replace("_", "")

In [ ]:
holmes = tokenize(gutenberg(1661).lower())
len(holmes)

In [ ]:
holmes_cfd = nltk.ConditionalFreqDist(nltk.bigrams(holmes))
holmes_cfd["the"].most_common(5)

#### activity 1

In [ ]:
random.randint(1, 6)

#### sampling

In [ ]:
row = holmes_cfd["holmes"]
words = list(row)
weights = [row[w] for w in words]
random.choices(words, weights=weights)[0]

In [ ]:
def sample(row, temperature=1.0):
    words = list(row)
    weights = [row[w] ** (1 / temperature) for w in words]
    return random.choices(words, weights=weights)[0]

In [ ]:
def generate(model, start, n=15, temperature=1.0):
    words = [start]
    while len(words) < n and words[-1] != "</s>":
        words.append(sample(model[words[-1]], temperature))
    return " ".join(w for w in words if w not in ("<s>", "</s>"))

generate(holmes_cfd, "the")

In [ ]:
def greedy(model, start, n=15):
    words = [start]
    while len(words) < n and words[-1] != "</s>":
        words.append(model[words[-1]].max())
    return " ".join(w for w in words if w not in ("<s>", "</s>"))

greedy(holmes_cfd, "the")

#### activity 2

In [ ]:
novels = {"alice": 11, "pride": 1342, "shakespeare": 100, "odyssey": 1727}
tokens = tokenize(gutenberg(novels["alice"]).lower())
novel = nltk.ConditionalFreqDist(nltk.bigrams(tokens))
len(tokens), len(set(tokens))

In [ ]:
for i in range(5):
    print(generate(novel, "the", 20))

#### temperature

In [ ]:
for t in [0.3, 1, 3]:
    print(t, generate(holmes_cfd, "the", 15, t))

#### activity 3

In [ ]:
lines = open("tweets_trump.txt", encoding="utf-8").read().splitlines()
tokens = []
for line in lines:
    tokens += ["<s>"] + tokenize(line) + ["</s>"]
tweets = nltk.ConditionalFreqDist(nltk.bigrams(tokens))
len(lines), len(tokens)

In [ ]:
greedy(tweets, "<s>", 40)

In [ ]:
for i in range(10):
    print(generate(tweets, "<s>", 40, temperature=1))

In [ ]:
for i in range(10):
    print(generate(tweets, "<s>", 40, temperature=0.5))

In [ ]:
for i in range(10):
    print(generate(tweets, "<s>", 40, temperature=2))

In [ ]:
fake = "The Democrats are doing a great honor to the United States ."
any(fake == " ".join(tokenize(line)) for line in lines)